# 02 - Limpeza dos Dados

Objetivo: tipar corretamente o dataset, criar features derivadas comportamentais e gravar a base pronta em `data/processed/`, sem descartar extremos que carregam sinal de fraude.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.cleaning import (
    clean_transactions,
    load_raw,
    report_missing,
    save_processed,
)

RAW_PATH = ROOT / "data" / "raw" / "transactions.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "transactions_clean.csv"

## Carregar bruto e checar qualidade

In [2]:
raw = load_raw(RAW_PATH)
print(raw.shape)
display(report_missing(raw) if not report_missing(raw).empty else "Sem valores ausentes.")
print("Duplicatas:", raw.duplicated().sum())

(299695, 17)


'Sem valores ausentes.'

Duplicatas: 0


## Decisões de limpeza

| Item | Decisão | Motivo |
| --- | --- | --- |
| Nulos / duplicatas | Nenhuma imputação necessária | Dataset já consistente |
| `transaction_id`, `user_id` | Remover da base de modelagem | Identificadores, não features |
| `transaction_time` | Converter para datetime UTC | Base para hour / day_of_week |
| Flags binárias | `int8` | Tipagem explícita |
| Extremos de `amount` / distância | **Manter** | Correlacionam com fraude (~40%+ no top 1%) |
| Features novas | mismatch país/BIN, razão amount/média, hora, noite | Sinais comportamentais da EDA |

In [3]:
clean = clean_transactions(raw)
clean.head()

,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud,hour,day_of_week,is_night,country_bin_mismatch,amount_to_avg_ratio
0,141,47,147.93,84.75,FR,FR,web,travel,0,1,1,1,2024-01-06 04:09:39+00:00,370.95,0,4,5,1,0,0.572906
1,141,47,147.93,107.90,FR,FR,web,travel,0,0,0,0,2024-01-09 20:13:47+00:00,149.62,0,20,1,0,0,0.729399
2,141,47,147.93,92.36,FR,FR,app,travel,1,1,1,1,2024-01-12 06:20:11+00:00,164.08,0,6,4,0,0,0.624349
3,141,47,147.93,112.47,FR,FR,web,fashion,0,1,1,1,2024-01-15 17:00:04+00:00,397.40,0,17,0,0,0,0.760292
4,141,47,147.93,132.91,FR,US,web,electronics,0,1,1,1,2024-01-17 01:27:31+00:00,935.28,0,1,2,1,1,0.898465


In [4]:
print("Colunas finais:", list(clean.columns))
print("Shape:", clean.shape)
print("Taxa country_bin_mismatch:", f"{clean['country_bin_mismatch'].mean():.2%}")
print(
    "Fraude com mismatch:",
    f"{clean.loc[clean['country_bin_mismatch'] == 1, 'is_fraud'].mean():.2%}",
)
clean[["amount", "avg_amount_user", "amount_to_avg_ratio", "shipping_distance_km"]].describe().T

Colunas finais: ['account_age_days', 'total_transactions_user', 'avg_amount_user', 'amount', 'country', 'bin_country', 'channel', 'merchant_category', 'promo_used', 'avs_match', 'cvv_result', 'three_ds_flag', 'transaction_time', 'shipping_distance_km', 'is_fraud', 'hour', 'day_of_week', 'is_night', 'country_bin_mismatch', 'amount_to_avg_ratio']
Shape: (299695, 20)
Taxa country_bin_mismatch: 7.92%
Fraude com mismatch: 11.28%


,count,mean,std,min,25%,50%,75%,max
amount,299695.0,177.165279,306.926507,1.000000,42.100000,89.99000,191.110000,16994.740000
avg_amount_user,299695.0,148.142973,200.364624,3.520000,46.190000,90.13000,173.450000,4565.290000
amount_to_avg_ratio,299695.0,1.317971,4.188530,0.001276,0.713657,1.00213,1.408299,597.512605
shipping_distance_km,299695.0,357.049028,427.672074,0.000000,136.600000,273.02000,409.180000,3748.560000


## Persistir base processada

In [5]:
saved = save_processed(clean, PROCESSED_PATH)
print(f"Salvo em: {saved}")
print(f"Tamanho: {saved.stat().st_size / 1e6:.1f} MB")
pd.read_csv(saved, nrows=3)

Salvo em: /Users/fiddelis/Projects/C318-Fraud-Detection/data/processed/transactions_clean.csv
Tamanho: 32.6 MB


,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud,hour,day_of_week,is_night,country_bin_mismatch,amount_to_avg_ratio
0,141,47,147.93,84.75,FR,FR,web,travel,0,1,1,1,2024-01-06 04:09:39+00:00,370.95,0,4,5,1,0,0.572906
1,141,47,147.93,107.90,FR,FR,web,travel,0,0,0,0,2024-01-09 20:13:47+00:00,149.62,0,20,1,0,0,0.729399
2,141,47,147.93,92.36,FR,FR,app,travel,1,1,1,1,2024-01-12 06:20:11+00:00,164.08,0,6,4,0,0,0.624349
